In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# 1. Load Dataset
dataset = load_dataset("imdb")

# Use small subset for learning
small_train = dataset["train"].select(range(1000))
small_test = dataset["test"].select(range(200))

# 2. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

# 3. Tokenization Function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length"
    )

# 4. Tokenize Dataset
train_dataset = small_train.map(
    tokenize_function,
    batched=True
)

test_dataset = small_test.map(
    tokenize_function,
    batched=True
)

# 5. Load Model
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# 6. Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch"
)

# 7. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

# 8. Train
trainer.train()

# 9. Evaluate
results = trainer.evaluate()

print(results)

In [ ]:
text = "This movie was fantastic and amazing"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

outputs = model(**inputs)

print(outputs.logits)

In [ ]:
import torch

probs = torch.softmax(outputs.logits, dim=1)

print(probs)